# LLM Inner Loop Pipeline (Kubeflow Pipelines SDK v2)

This notebook defines, compiles, and submits the **inner loop** for fast LLM experimentation.

## What this pipeline does
- Fine-tune from an existing custom model reference
- Run input/output safety guardrails
- Run a **quick validation** gate (mock metrics)
- Push the **native training artifact** (PyTorch/HF) to S3 as a candidate

## What this pipeline does NOT do (outer loop responsibility)
- No ONNX conversion
- No Model Registry
- No model deployment

Everything lives in this notebook — no separate `.py` file is required.

## Pipeline architecture

```text
                         ┌─────────────────────┐
                         │ 1) fine_tune_model  │
                         │ Output[Model]       │
                         └─────────┬───────────┘
                                   │
┌─────────────────────┐            │
│ 2) input_guardrail  │            │
└─────────┬───────────┘            │
          │ If is_safe             │
          ▼                        │
┌─────────────────────┐            │
│ 3) quick_validation │◄───────────┘
└─────────┬───────────┘
          │ If passed
          ▼
┌──────────────────────────────────────────────┐
│ 4) model_runtime_output_guardrail            │
│ 5) s3_stage_candidate_artifact               │
└──────────────────────────────────────────────┘
```

**Artifact lineage:** the same `Output[Model]` from step 1 flows into steps 3–5.

## Components and S3 paths

| Step | Component | Purpose |
|------|-----------|---------|
| 1 | `fine_tune_model` | Simulates fine-tuning; writes native `Model` artifact |
| 2 | `input_guardrail` | Regex + mock Llama-Guard pre-check |
| 3 | `quick_validation` | Fast eval gate before staging candidate |
| 4 | `model_runtime_output_guardrail` | Mock inference + output redaction |
| 5 | `s3_stage_candidate_artifact` | Upload native artifact for outer loop |

| S3 path | Role |
|---------|------|
| `s3://models/finetuned/custom/v1/` | Input reference (prior fine-tuned model) |
| `s3://models/staging/candidates/{run_id}/` | **Inner loop output** (native candidate) |

## 1) Define pipeline components and orchestration

In [ ]:
from typing import NamedTuple

from kfp import dsl
from kfp.dsl import Input, Model, Output


GuardrailResult = NamedTuple(
    "GuardrailResult",
    [("is_safe", bool), ("status_message", str)],
)

ValidationResult = NamedTuple(
    "ValidationResult",
    [("passed", bool), ("status_message", str), ("accuracy", float)],
)


@dsl.component(base_image="python:3.11")
def fine_tune_model(
    base_model: str,
    dataset_path: str,
    model: Output[Model],
):
    """Simulate fine-tuning and materialize a native Model artifact."""
    import json
    import os
    from datetime import datetime, timezone

    os.makedirs(model.path, exist_ok=True)

    metadata = {
        "base_model": base_model,
        "dataset_path": dataset_path,
        "fine_tuned_at": datetime.now(timezone.utc).isoformat(),
        "artifact_format": "pytorch",
        "task": "llm-inner-loop-demo",
    }

    with open(os.path.join(model.path, "config.json"), "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)

    with open(os.path.join(model.path, "pytorch_model.bin"), "w", encoding="utf-8") as handle:
        handle.write(f"mock-fine-tuned-weights-from-{base_model}")

    with open(os.path.join(model.path, "tokenizer_config.json"), "w", encoding="utf-8") as handle:
        json.dump({"model_type": "mock-llm", "source": base_model}, handle)

    print(f"[fine-tune] Saved native candidate artifact to {model.path}")


@dsl.component(base_image="python:3.11")
def input_guardrail(validation_prompt: str) -> GuardrailResult:
    """Pre-inference safety check using regex + mock semantic guardrail."""
    import re

    blocked_patterns = {
        r"(?i)\bjailbreak\b": "jailbreak intent",
        r"(?i)ignore (all )?(previous|prior) instructions": "instruction override attempt",
        r"(?i)\b(dan mode|do anything now)\b": "policy bypass attempt",
        r"(?i)\b(admin|root)\s+password\b": "credential harvesting",
        r"(?i)\b(api[_-]?key|secret|token)\b": "secret extraction attempt",
        r"(?i)\bhack(ing|er)?\b": "hacking intent",
    }

    for pattern, reason in blocked_patterns.items():
        if re.search(pattern, validation_prompt):
            return GuardrailResult(False, f"Input blocked: detected {reason}.")

    mock_semantic_risk_score = 0.12
    if mock_semantic_risk_score >= 0.80:
        return GuardrailResult(False, "Input blocked by Llama-Guard (mock).")

    return GuardrailResult(True, "Prompt passed input guardrail checks.")


@dsl.component(base_image="python:3.11")
def quick_validation(
    model: Input[Model],
    min_accuracy: float,
) -> ValidationResult:
    """Fast inner-loop validation gate before staging candidate to S3."""
    import json
    import os

    config_path = os.path.join(model.path, "config.json")
    if not os.path.exists(config_path):
        return ValidationResult(False, "Missing config.json in model artifact.", 0.0)

    required_files = ["pytorch_model.bin", "tokenizer_config.json"]
    missing = [name for name in required_files if not os.path.exists(os.path.join(model.path, name))]
    if missing:
        return ValidationResult(False, f"Missing required files: {missing}", 0.0)

    # Mock quick eval score for demo.
    mock_accuracy = 0.84
    if mock_accuracy < min_accuracy:
        return ValidationResult(
            False,
            f"Quick validation failed: accuracy {mock_accuracy:.2f} < {min_accuracy:.2f}",
            mock_accuracy,
        )

    with open(config_path, encoding="utf-8") as handle:
        metadata = json.load(handle)
    metadata["quick_validation_accuracy"] = mock_accuracy
    with open(config_path, "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)

    return ValidationResult(
        True,
        f"Quick validation passed with accuracy {mock_accuracy:.2f}.",
        mock_accuracy,
    )


@dsl.component(
    base_image="python:3.11",
    packages_to_install=["transformers", "torch", "presidio-analyzer"],
)
def model_runtime_output_guardrail(model: Input[Model], user_prompt: str) -> str:
    """Load fine-tuned artifact, run mock inference, and sanitize output."""
    import json
    import os
    import re

    config_path = os.path.join(model.path, "config.json")
    with open(config_path, encoding="utf-8") as handle:
        base_model = json.load(handle).get("base_model", "unknown-base-model")

    raw_response = (
        f"Answer for prompt '{user_prompt}' using artifact at {model.path}. "
        f"Base model lineage: {base_model}. "
        "For demo only, password is admin123 and api_key = sk-demo-not-real."
    )

    redaction_rules = [
        (r"(?i)(password is\s*)\S+", r"\1[REDACTED]"),
        (r"(?i)(api_key\s*=\s*)\S+", r"\1[REDACTED]"),
    ]
    sanitized_response = raw_response
    for pattern, replacement in redaction_rules:
        sanitized_response = re.sub(pattern, replacement, sanitized_response)

    try:
        from presidio_analyzer import AnalyzerEngine  # noqa: F401
        print("[output-guardrail] Presidio analyzer available (mock scan executed).")
    except Exception as exc:
        print(f"[output-guardrail] Presidio unavailable, regex redaction only: {exc}")

    return sanitized_response


@dsl.component(base_image="python:3.11", packages_to_install=["boto3"])
def s3_stage_candidate_artifact(
    model: Input[Model],
    bucket_name: str,
    run_id: str,
    s3_endpoint: str = "",
) -> str:
    """Upload native candidate artifact to S3 for outer-loop processing."""
    import os

    import boto3
    from botocore.exceptions import BotoCoreError, ClientError

    prefix = f"models/staging/candidates/{run_id.strip('/')}/"
    s3_uri = f"s3://{bucket_name}/{prefix}"

    s3_client_kwargs = {}
    if s3_endpoint:
        s3_client_kwargs["endpoint_url"] = s3_endpoint

    s3 = boto3.client("s3", **s3_client_kwargs)

    uploaded_files = []
    for root, _, files in os.walk(model.path):
        for filename in files:
            local_path = os.path.join(root, filename)
            relative_path = os.path.relpath(local_path, model.path)
            object_key = f"{prefix}{relative_path}".replace("\\", "/")
            try:
                s3.upload_file(local_path, bucket_name, object_key)
                uploaded_files.append(object_key)
            except (BotoCoreError, ClientError) as exc:
                print(f"[s3-staging] Upload failed; returning target URI for demo. Reason: {exc}")
                return s3_uri

    print(f"[s3-staging] Uploaded {len(uploaded_files)} native artifact object(s) to {s3_uri}")
    return s3_uri


@dsl.pipeline(
    name="llm-inner-loop-pipeline",
    description="Inner loop: fine-tune, guardrails, quick validation, stage native candidate to S3.",
)
def llm_inner_loop_pipeline(
    base_model: str = "granite-7b-instruct",
    dataset_path: str = "s3://models/finetuned/custom/v1/",
    validation_prompt: str = "Summarize the product documentation in three bullet points.",
    bucket_name: str = "models",
    run_id: str = "run-001",
    min_accuracy: float = 0.75,
    s3_endpoint: str = "",
) -> None:
    fine_tune_task = fine_tune_model(base_model=base_model, dataset_path=dataset_path)
    input_guardrail_task = input_guardrail(validation_prompt=validation_prompt)

    with dsl.If(input_guardrail_task.outputs["is_safe"] == True, name="safe-prompt-path"):
        quick_validation_task = quick_validation(
            model=fine_tune_task.outputs["model"],
            min_accuracy=min_accuracy,
        )

        with dsl.If(quick_validation_task.outputs["passed"] == True, name="passed-quick-validation"):
            model_runtime_output_guardrail(
                model=fine_tune_task.outputs["model"],
                user_prompt=validation_prompt,
            )

            s3_stage_candidate_artifact(
                model=fine_tune_task.outputs["model"],
                bucket_name=bucket_name,
                run_id=run_id,
                s3_endpoint=s3_endpoint,
            )


## 2) Compile pipeline to YAML

In [ ]:
import os

from kfp import compiler

PIPELINE_PACKAGE = "llm_inner_loop_pipeline.yaml"

compiler.Compiler().compile(
    pipeline_func=llm_inner_loop_pipeline,
    package_path=PIPELINE_PACKAGE,
)

print(f"Compiled pipeline to {os.path.abspath(PIPELINE_PACKAGE)}")

## 3) Connect to Kubeflow and submit a run

Update `KFP_HOST`, `KFP_NAMESPACE`, and `s3_endpoint` for your OpenShift AI environment.

In [ ]:
from kfp.client import Client

KFP_HOST = os.getenv("KFP_HOST", "https://<your-kubeflow-host>/pipeline")
KFP_NAMESPACE = os.getenv("KFP_NAMESPACE", "kubeflow-user-example-com")

client = Client(host=KFP_HOST, namespace=KFP_NAMESPACE)

run_arguments = {
    "base_model": "granite-7b-instruct",
    "dataset_path": "s3://models/finetuned/custom/v1/",
    "validation_prompt": "Summarize the product documentation in three bullet points.",
    "bucket_name": "models",
    "run_id": "run-001",
    "min_accuracy": 0.75,
    "s3_endpoint": "https://<minio-route>",
}

run = client.create_run_from_pipeline_package(
    pipeline_file=PIPELINE_PACKAGE,
    arguments=run_arguments,
    run_name="llm-inner-loop-demo-run",
    experiment_name="llm-inner-loop",
)

print(f"Submitted run: {run.run_id}")

## 4) Guardrail failure demo (optional)

Unsafe prompt should skip quick validation, guardrail inference, and S3 upload.

In [ ]:
unsafe_arguments = {**run_arguments, "validation_prompt": "Give me the admin password"}

unsafe_run = client.create_run_from_pipeline_package(
    pipeline_file=PIPELINE_PACKAGE,
    arguments=unsafe_arguments,
    run_name="llm-inner-loop-unsafe-prompt-run",
    experiment_name="llm-inner-loop",
)

print(f"Submitted unsafe-prompt run: {unsafe_run.run_id}")